# MNIST MLP Pure Inference on PYNQ-Z2
Run this notebook **on the PYNQ-Z2 board** (Jupyter at 192.168.2.99)

Requirements:
- `mlp.bit` + `mlp.hwh` in the same directory as this notebook
- `fpga_weights/` folder (w1.npy … b4.npy) in the same directory
- An image file named `test_image.png` (e.g. a hand-drawn digit)


In [ ]:
import numpy as np
import time
from PIL import Image
from pynq import Overlay, allocate
import pynq.lib.dma


In [ ]:
# Load the FPGA overlay (bitstream + hardware handoff)
print('Loading overlay...')
ol = Overlay('./mlp.bit')
print('Overlay loaded.')
dma = ol.axi_dma_0
mlp = ol.mlp_top_0


In [ ]:
# Load quantised weights (int16) and allocate contiguous DDR buffers
SCALE = 1024
weight_bufs = {}
bias_bufs   = {}

for k in range(1, 5):
    w = np.load(f'fpga_weights/w{k}.npy')
    b = np.load(f'fpga_weights/b{k}.npy')

    w_buf = allocate(shape=w.shape, dtype=np.int16)
    b_buf = allocate(shape=b.shape, dtype=np.int16)
    w_buf[:] = w
    b_buf[:] = b
    weight_bufs[k] = w_buf
    bias_bufs[k]   = b_buf

REG = {
    'w1': 0x10, 'b1': 0x18,
    'w2': 0x20, 'b2': 0x28,
    'w3': 0x30, 'b3': 0x38,
    'w4': 0x40, 'b4': 0x48,
}
for k in range(1, 5):
    mlp.write(REG[f'w{k}'], weight_bufs[k].physical_address & 0xFFFFFFFF)
    mlp.write(REG[f'b{k}'], bias_bufs[k].physical_address   & 0xFFFFFFFF)

print('Weights configured in FPGA registers.')


In [ ]:
def preprocess(img_uint8):
    x = img_uint8.astype(np.float32) / 255.0
    x = (x - 0.1307) / 0.3081
    return np.clip(np.round(x * SCALE), -32768, 32767).astype(np.int16)

in_buf  = allocate(shape=(784,), dtype=np.int16)
out_buf = allocate(shape=(10,),  dtype=np.int16)

def run_fpga_inference(img_array):
    in_buf[:] = preprocess(img_array.reshape(784))
    mlp.write(0x00, 1)
    dma.sendchannel.transfer(in_buf)
    dma.recvchannel.transfer(out_buf)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    return out_buf[:].astype(np.float32) / SCALE


In [ ]:
# Load your own test image (e.g. upload 'test_image.png' to Jupyter)
try:
    img = Image.open('test_image.png').convert('L').resize((28, 28))
    img_array = np.array(img)
    
    # Run Hardware Inference
    logits = run_fpga_inference(img_array)
    pred = np.argmax(logits)
    
    print(pred)
except FileNotFoundError:
    print('Please upload a test_image.png file to the same directory.')


In [ ]:
for k in range(1,5):
    weight_bufs[k].freebuffer()
    bias_bufs[k].freebuffer()
in_buf.freebuffer()
out_buf.freebuffer()
